# Sweeps

Every request is a sweep. A concrete artifact is a sweep of one, a `Sweep` of artifacts is a group, and an artifact constructed with a `Sweep` in any parameter or dependency stands for the Cartesian product of all of them. The same three calls take every one of these:

- `expand(x)`: the flat list of concrete artifacts `x` stands for
- `resolve(x)`: everything they are built from, dependencies first, each path once
- `lab.declare(x, root=...)`: a preview, or with `commit=True` the written manifests

These cells declare into a local temporary folder. Nothing is downloaded and no job runs.

In [1]:
from collections import Counter
from dataclasses import replace
from pathlib import Path
from tempfile import mkdtemp

import lab
from artifacts.core.resolve import resolve
from artifacts.core.SGD.training import LoopConfig, LRSchedule, OptimizerParameters, TrainingParameters
from artifacts.core.sweep import Sweep, expand
from artifacts.dataset import DataSet
from artifacts.sources import SourceURL
from artifacts.stages.pretraining import Pretraining
from artifacts.tokenizers.bpe import Tokenizer
from models.transformer import Transformer

## A concrete artifact is a sweep of one

In [2]:
odyssey = SourceURL(name="odyssey", url="https://www.gutenberg.org/cache/epub/1727/pg1727.txt")
moby_dick = SourceURL(name="moby-dick", url="https://www.gutenberg.org/cache/epub/2701/pg2701.txt")

assert expand(odyssey) == [odyssey]
resolve(odyssey)

[SourceURL(name='odyssey', url='https://www.gutenberg.org/cache/epub/1727/pg1727.txt')]

## A group is a sweep

In [3]:
books = Sweep((odyssey, moby_dick))
[str(a.artifact_path) for a in expand(books)]

['sources/odyssey', 'sources/moby-dick']

## Every combination needs its own path

A swept parameter has to be part of the artifact's identity. A `SourceURL` lives at `sources/<name>`, so sweeping only the `url` puts two different definitions at `sources/hello`, and resolution refuses before anything touches storage.

In [4]:
try:
    resolve(SourceURL(name="hello", url=Sweep((odyssey.url, moby_dick.url))))
except ValueError as error:
    print(error)

different definitions at sources/hello: roots[0] and roots[1]


## Sweep dependencies and parameters together

One pretraining declaration with a swept dependency (which book the dataset is built from) and two swept model parameters. Sweeps are independent, so it stands for 2 × 3 × 2 = 12 runs.

`vocab_size` stays fixed: the model's must equal the tokenizer's, and sweeps are never linked, so sweeping both would produce mismatched combinations that fail to construct.

In [5]:
tokenizer = Tokenizer(1000, ("<pad>", "<|endoftext|>"), sources=(odyssey, moby_dick))
datasets = tuple(DataSet.from_sources(tokenizer, (book,), (book,)) for book in (odyssey, moby_dick))

model = Transformer(
    vocab_size=1000,
    sequence_length=256,
    d_model=Sweep((128, 256, 512)),
    num_layers=Sweep((4, 8)),
    d_ff=1024,
    num_heads=8,
    rope_theta=10_000,
)
training_parameters = TrainingParameters(
    total_steps=1000,
    batch_size=32,
    max_norm=1.0,
    lr_schedule=LRSchedule(3e-4, 3e-5, 100, 1000),
    optimizer="torch.optim.AdamW",
    optimizer_parameters=OptimizerParameters(3e-4, (0.9, 0.95), 0.1, 1e-8),
)
loop_config = LoopConfig(val_every=100, gpu_check_every=100)

pretraining = Pretraining(
    run_id="sweep-demo",
    dataset=Sweep(datasets),
    model=model,
    training_parameters=training_parameters,
    loop_config=loop_config,
)
type(pretraining).__name__

'ArtifactSweep'

In [6]:
runs = expand(pretraining)
print(len(runs), "runs")
for run in runs:
    book = run.dataset.train_set[0].source.name
    p = run.model
    print(f"{book:10} d_model={p.d_model:<4} layers={p.num_layers}  {run.artifact_path}")

12 runs
odyssey    d_model=128  layers=4  runs/sweep-demo/pretraining/0-1000-50ade0d057
odyssey    d_model=128  layers=8  runs/sweep-demo/pretraining/0-1000-989946df9b
odyssey    d_model=256  layers=4  runs/sweep-demo/pretraining/0-1000-267d44dacf
odyssey    d_model=256  layers=8  runs/sweep-demo/pretraining/0-1000-ae2072946c
odyssey    d_model=512  layers=4  runs/sweep-demo/pretraining/0-1000-2137559f0d
odyssey    d_model=512  layers=8  runs/sweep-demo/pretraining/0-1000-a19889d69f
moby-dick  d_model=128  layers=4  runs/sweep-demo/pretraining/0-1000-af2926d4bd
moby-dick  d_model=128  layers=8  runs/sweep-demo/pretraining/0-1000-8b9d041373
moby-dick  d_model=256  layers=4  runs/sweep-demo/pretraining/0-1000-f87384fa82
moby-dick  d_model=256  layers=8  runs/sweep-demo/pretraining/0-1000-e35e025d6a
moby-dick  d_model=512  layers=4  runs/sweep-demo/pretraining/0-1000-02e6ddb351
moby-dick  d_model=512  layers=8  runs/sweep-demo/pretraining/0-1000-ec909a51b0


The graph holds each shared dependency once: two sources, one tokenizer, two tokenized sources, two datasets, and the twelve runs.

In [7]:
graph = resolve(pretraining)
Counter(type(a).__name__ for a in graph)

Counter({'Pretraining': 12,
         'SourceURL': 2,
         'TokenizedSource': 2,
         'DataSet': 2,
         'Tokenizer': 1})

## Declare locally

The preview writes nothing; every row is new.

In [8]:
root = Path(mkdtemp(prefix="launchpad-sweeps-"))
preview = lab.declare(pretraining, root=root)
assert list(root.iterdir()) == []

sources/odyssey                                new
sources/moby-dick                              new
tokenizers/bpe-1000-514731ec25                 new
tokenized/bpe-1000-514731ec25/odyssey          new
datasets/dataset-64fa907865                    new
runs/sweep-demo/pretraining/0-1000-50ade0d057  new
runs/sweep-demo/pretraining/0-1000-989946df9b  new
runs/sweep-demo/pretraining/0-1000-267d44dacf  new
runs/sweep-demo/pretraining/0-1000-ae2072946c  new
runs/sweep-demo/pretraining/0-1000-2137559f0d  new
runs/sweep-demo/pretraining/0-1000-a19889d69f  new
tokenized/bpe-1000-514731ec25/moby-dick        new
datasets/dataset-9a4b8019c7                    new
runs/sweep-demo/pretraining/0-1000-af2926d4bd  new
runs/sweep-demo/pretraining/0-1000-8b9d041373  new
runs/sweep-demo/pretraining/0-1000-f87384fa82  new
runs/sweep-demo/pretraining/0-1000-e35e025d6a  new
runs/sweep-demo/pretraining/0-1000-02e6ddb351  new
runs/sweep-demo/pretraining/0-1000-ec909a51b0  new

19 new
ok -- 19 to declare


`commit=True` writes one manifest per unique path. It declares the jobs; it does not run them.

In [9]:
report = lab.declare(pretraining, root=root, commit=True)
len(list(root.rglob("manifest.json")))

sources/odyssey                                declared  (created)
sources/moby-dick                              declared  (created)
tokenizers/bpe-1000-514731ec25                 declared  (created)
tokenized/bpe-1000-514731ec25/odyssey          declared  (created)
datasets/dataset-64fa907865                    declared  (created)
runs/sweep-demo/pretraining/0-1000-50ade0d057  declared  (created)
runs/sweep-demo/pretraining/0-1000-989946df9b  declared  (created)
runs/sweep-demo/pretraining/0-1000-267d44dacf  declared  (created)
runs/sweep-demo/pretraining/0-1000-ae2072946c  declared  (created)
runs/sweep-demo/pretraining/0-1000-2137559f0d  declared  (created)
runs/sweep-demo/pretraining/0-1000-a19889d69f  declared  (created)
tokenized/bpe-1000-514731ec25/moby-dick        declared  (created)
datasets/dataset-9a4b8019c7                    declared  (created)
runs/sweep-demo/pretraining/0-1000-af2926d4bd  declared  (created)
runs/sweep-demo/pretraining/0-1000-8b9d041373  declared  (crea

19

## A group of sweeps

Groups nest: a `Sweep` of declarations concatenates their runs. Here the declared sweep is grouped with a learning-rate sweep on the smallest model. The sources, tokenizer and datasets are already declared and are reused; only the three new runs are created.

In [10]:
lr_sweep = Pretraining(
    run_id="lr-demo",
    dataset=datasets[0],
    model=replace(model, d_model=128, num_layers=4),
    training_parameters=replace(
        training_parameters,
        lr_schedule=replace(training_parameters.lr_schedule, max_learning_rate=Sweep((1e-4, 3e-4, 1e-3))),
    ),
    loop_config=loop_config,
)
group = Sweep((pretraining, lr_sweep))
print(len(expand(group)), "runs in the group")

report = lab.declare(group, root=root, commit=True)
print(sum(row["created"] for row in report.rows), "created")

15 runs in the group
sources/odyssey                                declared
sources/moby-dick                              declared
tokenizers/bpe-1000-514731ec25                 declared
tokenized/bpe-1000-514731ec25/odyssey          declared
datasets/dataset-64fa907865                    declared
runs/sweep-demo/pretraining/0-1000-50ade0d057  declared
runs/sweep-demo/pretraining/0-1000-989946df9b  declared
runs/sweep-demo/pretraining/0-1000-267d44dacf  declared
runs/sweep-demo/pretraining/0-1000-ae2072946c  declared
runs/sweep-demo/pretraining/0-1000-2137559f0d  declared
runs/sweep-demo/pretraining/0-1000-a19889d69f  declared
tokenized/bpe-1000-514731ec25/moby-dick        declared
datasets/dataset-9a4b8019c7                    declared
runs/sweep-demo/pretraining/0-1000-af2926d4bd  declared
runs/sweep-demo/pretraining/0-1000-8b9d041373  declared
runs/sweep-demo/pretraining/0-1000-f87384fa82  declared
runs/sweep-demo/pretraining/0-1000-e35e025d6a  declared
runs/sweep-demo/pretraining

Declaring the same group again changes nothing.

In [11]:
again = lab.declare(group, root=root, commit=True)
assert not any(row["created"] or row["updated"] for row in again.rows)
print(f"manifests under {root}; remove with shutil.rmtree(root)")

sources/odyssey                                declared
sources/moby-dick                              declared
tokenizers/bpe-1000-514731ec25                 declared
tokenized/bpe-1000-514731ec25/odyssey          declared
datasets/dataset-64fa907865                    declared
runs/sweep-demo/pretraining/0-1000-50ade0d057  declared
runs/sweep-demo/pretraining/0-1000-989946df9b  declared
runs/sweep-demo/pretraining/0-1000-267d44dacf  declared
runs/sweep-demo/pretraining/0-1000-ae2072946c  declared
runs/sweep-demo/pretraining/0-1000-2137559f0d  declared
runs/sweep-demo/pretraining/0-1000-a19889d69f  declared
tokenized/bpe-1000-514731ec25/moby-dick        declared
datasets/dataset-9a4b8019c7                    declared
runs/sweep-demo/pretraining/0-1000-af2926d4bd  declared
runs/sweep-demo/pretraining/0-1000-8b9d041373  declared
runs/sweep-demo/pretraining/0-1000-f87384fa82  declared
runs/sweep-demo/pretraining/0-1000-e35e025d6a  declared
runs/sweep-demo/pretraining/0-1000-02e6ddb351  d